# 모듈 3. 출력 다루기 - Output Parser

> LLM의 자유형 텍스트 응답을 타임이 보장된 Python 객체 (dict, list, Pydantic 모델 등)로 자동 변환하는 Output Parser를 실습한다.

**학습 목표**
- PydanticOutputParser로 스키마 기반 구조화 추출
- CommaSeparatedList / Structured / Json / Datetime / Enum Parser 활용
- OutputFixingParser로 파싱 실패 시 LLM 자동 보정
- Custom OutputParser 직접 구현

## 1. PydanticOutputParser

- Pydantic: Python 데이터 검증 & 설정 관리 라이브러리
    - "데이터를 타입 힌트 기반으소 선언하고, 자동 검증·변환해주는 것"

In [2]:
from dotenv import load_dotenv, find_dotenv
import os

# .env 파일에서 환경변수 로드
load_dotenv(find_dotenv(), override=True)

True

In [3]:
# 입력 데이터
email_conversation = """From: 교육담당자 (edu@sw.or.kr)"
To: 강희숙 (coojugi@naver.com)
Subject: 교육 안내

문의사항이 있으신 경우 담당자에게 연락 바랍니다.

* 선착순 모집이며 조기 마감될 수 있습니다.
* 오프라인과 온라인 수업이 동시에 있는 경우, 동시 접수가 불가능합니다.
* 8월 11일(월) LLM 온라인 교육이 있습니다.

그외의 교육은 오프라인 교육입니다.

* 온라인으로 교육 참여하시는 경우 교재 파일 링크를 제공해드리며 별도 실물 교재는 제공해 드리지 않습니다.
"""

In [4]:
from langchain_openai import ChatOpenAI

# 1. LLM 초기화
llm = ChatOpenAI(
    model = 'gpt-4o-mini',
    temperature = 0
)

In [7]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "다음의 이메일 내용 중 중요한 내용을 추출해 주세요.\n\n{email_conversation}"
)

chain = prompt | llm # 정보를 llm에 전달
response = chain.invoke({"email_conversation": email_conversation})
print(response.content)

중요한 내용 요약:

- 선착순 모집, 조기 마감 가능
- 오프라인과 온라인 수업 동시 접수 불가
- 8월 11일(월) LLM 온라인 교육 예정
- 온라인 교육 참여 시 교재 파일 링크 제공, 실물 교재는 제공되지 않음


In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# 2. 출력 스키마 정의 (Pydantic 모델) : 모델이 채워야 할 필드/설명을 명확히 지정
class EmailSummary(BaseModel): # BaseModel을 상속받아 내가 원하는 구조 형태로 새롭게 정의
    person: str = Field(description="메일을 보낸 사람의 이름")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일의 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 날짜")

# 3. 파서 준비 : LLM의 응답 텍스트를 EmailSummary로 검증/파싱
parser = PydanticOutputParser(pydantic_object=EmailSummary) # EmailSummry를 가져와 출력의 format을 정의
print(parser.get_format_instructions()) # LLM에게 출력 형식에 대한 지침을 제공

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "메일을 보낸 사람의 이름", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일의 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 날짜", "title": "Date", "type": "string"}}, "required": ["person", "email", "subject", "summary", "date"]}
```


In [10]:
# 4. 프롬프트 템플릿 정의 : {format} 자리에 parser.get_format_instructions() 삽입
prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the following questions in KOREAN.

QUESTION:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT:
{format}
"""
)

# 5. 포맷 지시 삽입 (partial)
prompt = prompt.partial(format=parser.get_format_instructions())

In [11]:
# 6. 체인 구성 (Prompt -> LLM -> Parser): 최종적으로 Pydantic 객체(EmailSummary)
pipline = prompt | llm | parser

# 7. 실행 (invoke): question과 email_conversation을 넣어 호출
response = pipline.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 중요한 내용을 추출해 주세요."
    }
)

In [12]:
# 8. 결과 출력: Pydatic v2(model_dump) / v1(dict) 호환 출력
print(response)

try:
    print(response.model_dump())
except AttributeError:
    print(response.dict())

person='교육담당자' email='edu@sw.or.kr' subject='교육 안내' summary='선착순 모집이며 조기 마감될 수 있습니다. 오프라인과 온라인 수업이 동시에 있는 경우, 동시 접수가 불가능합니다. 8월 11일(월) LLM 온라인 교육이 있습니다. 온라인으로 교육 참여 시 교재 파일 링크 제공, 실물 교재는 제공되지 않습니다.' date='8월 11일(월)'
{'person': '교육담당자', 'email': 'edu@sw.or.kr', 'subject': '교육 안내', 'summary': '선착순 모집이며 조기 마감될 수 있습니다. 오프라인과 온라인 수업이 동시에 있는 경우, 동시 접수가 불가능합니다. 8월 11일(월) LLM 온라인 교육이 있습니다. 온라인으로 교육 참여 시 교재 파일 링크 제공, 실물 교재는 제공되지 않습니다.', 'date': '8월 11일(월)'}


## 2. CommaSeparatedListOutputParser

In [1]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

output_parser = CommaSeparatedListOutputParser()

format_instructions = output_parser.get_format_instructions()
format_instructions

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

In [ ]:
prompt = PromptTemplate(
    template = "List 10 {subject}. \n{format_instructions}",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions} # 이 전 셀에서 있었던 형식을 끼워넣음
)

model = ChatOpenAI(temperature=0, model='gpt-4o-mini')

chain = prompt | model | output_parser

In [4]:
chain.invoke({"subject": "미국 국립공원"})

['옐로스톤',
 '그랜드 캐니언',
 '요세미티',
 '자이언',
 '스모키 마운틴',
 '아카디아',
 '세쿼이아',
 '브라이스 캐니언',
 '올드 포레스트',
 '에버글레이드']

## 3. JsonOutputParser

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

In [7]:
model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

class Topic(BaseModel):
    description: str = Field(description="주제에 대한 간결한 설명")
    keywords: str = Field(description="설명에 대한 주요 키워드(2개 이상)")

question = "도널드 트럼프의 외교정책에 대해 설명해주세요."

parser = JsonOutputParser(pydantic_object=Topic)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간단하게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n #Question: {question}")
    ]
)
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | model | parser

chain.invoke({"question": question})

{'description': "도널드 트럼프의 외교정책은 '미국 우선주의'를 중심으로 하며, 무역 협정 재협상, NATO 동맹국에 대한 방위비 분담 요구, 이란 핵 합의 탈퇴, 중국과의 무역 전쟁 등이 포함된다.",
 'keywords': '미국 우선주의, 무역 전쟁'}

In [10]:
question = "도널드 트럼프의 외교정책에 대해서 설명해주세요."

parser = parser = JsonOutputParser() # 특정 key 형식을 지정하지 않음

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간단하게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n #Question: {question}")
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | model | parser

response = chain.invoke({"question": question})

print(response)

{'도널드_트럼프의_외교정책': {'주요_특징': ['미국 우선주의: 미국의 이익을 최우선으로 고려', '무역전쟁: 중국과의 무역 갈등을 통해 관세 인상', '북한과의 대화: 김정은과의 정상회담을 통한 비핵화 협상 시도', 'NATO 재조정: NATO 동맹국들에게 방위비 분담 증액 요구', '중동 정책: 이스라엘과의 관계 강화 및 이란 핵 협정 탈퇴'], '비판': ['국제 동맹 약화: 전통적인 동맹국들과의 관계 악화', '일방주의: 다자간 협력보다는 단독 행동 선호', '인권 문제 무시: 특정 국가와의 관계에서 인권 문제 경시']}}


## 4. Custion Output Parser

In [11]:
from langchain_core.output_parsers import BaseOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI


In [13]:
class CommaSeperatedListOutputParser(BaseOutputParser):
    """
    쉼표로 구분된 항목 목록을 파싱하는 간단한 출력 파서
    """
    def parse(self, text: str) -> list:
        cleaned_text = text.strip()
        if not cleaned_text:
            return []
        items = [item.strip() for item in cleaned_text.split(",")]
        return items

template = """
다음 주제에 관련된 항목을 5개로 내열해주세요: {topic}

항목은 쉼표로 구분해 주세요.
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["topic"]
)

output_parser = CommaSeperatedListOutputParser()

In [14]:
import os

llm = ChatOpenAI(temperature=0, model=os.getenv("OPENAI_DEFAULT_MODEL", "gpt-4o-mini"))
chain = prompt | llm | output_parser

result = chain.invoke({"topic": "인공지능의 응용 분야"})
print(result)
print(type(result))

['자율주행차', '의료 진단', '자연어 처리', '이미지 인식', '추천 시스템']
<class 'list'>
